## Historisk rekonstruktion av grundvattennivå tidsserier med sparade kalibreringsinställningar

Denna notebook återanvänder en kalibrerad modellkonfiguration från kalibrerings notebooken och genomför historisk rekonstruktion över hela tillgängliga period.

Steg:
1. Importera bibliotek
2. Läs in sparade kalibreringsinställningar och indata
3. Bygg modell från sparade inställningar
4. Historisk rekonstruktion
5. Exportera resultat till CSV och Excel

### 0. Importera bibliotek

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pastas as ps

### 1. Läs in sparade kalibreringsinställningar och indata

In [2]:
# Alla in- och utdatafiler i denna workflow ligger i ../data
data_dir = "../data"
settings_file = os.path.join(data_dir, "kalibrering_settings_FAO-24.json")

if not os.path.exists(settings_file):
    raise FileNotFoundError(f"Inställningsfil saknas: {settings_file}")

with open(settings_file, "r", encoding="utf-8") as f:
    settings = json.load(f)

params_file = os.path.join(data_dir, settings["parameter_file"])
if not os.path.exists(params_file):
    raise FileNotFoundError(f"Parameterfil saknas: {params_file}")
params = pd.read_csv(params_file, index_col=0)

prec = pd.read_csv(
    os.path.join(data_dir, "precip_data_km.csv"),
    index_col=0,
    parse_dates=[0],
    date_format="%Y-%m-%d"
).squeeze("columns")

temp = pd.read_csv(
    os.path.join(data_dir, "temp_data_km.csv"),
    index_col=0,
    parse_dates=[0],
    date_format="%Y-%m-%d"
).squeeze("columns")

evap = pd.read_csv(
    os.path.join(data_dir, "pet_data_km.csv"),
    index_col=0,
    parse_dates=[0],
    date_format="%Y-%m-%d"
)

evap_choice = settings["selected_pet_method"]
if evap_choice not in evap.columns:
    raise KeyError(f"PET-metoden '{evap_choice}' finns inte i pet_data_km.csv")
evap_selected = evap[evap_choice]

# Observationer används för jämförelse i plott om filen finns
ho = None
ho_file = os.path.join(data_dir, "head_abs_km_compensated_daily.xlsx")
if os.path.exists(ho_file):
    ho = pd.read_excel(ho_file, usecols=[0, 1], index_col=0, parse_dates=True).squeeze()
    ho.name = "ho"

print(f"Inställningsfil: {settings_file}")
print(f"Vald PET-metod: {evap_choice}")
print(f"Parametrar inlästa: {len(params)}")
print(f"Längd prec/temp/evap: {len(prec)}/{len(temp)}/{len(evap_selected)}")
print(f"Observationer inlästa: {ho is not None}")

Inställningsfil: ../data\kalibrering_settings_FAO-24.json
Vald PET-metod: FAO-24
Parametrar inlästa: 14
Längd prec/temp/evap: 10945/10942/10928
Observationer inlästa: True


### 2. Bygg modell från sparade inställningar

In [ ]:
if ho is None:
    raise ValueError(
        "Observationer krävs för att skapa modellen. Lägg head_abs_km_compensated_daily.xlsx i ../data."
    )

# Kompatibilitetsfallback för miljöer där pandas Day-offset inte hanteras av Pastas-hjälpfunktioner
import pastas.timeseries_utils as tsu
import pastas.model as ps_model
import pastas.timeseries as ps_timeseries

if not hasattr(tsu, "_dayfreq_patch_applied"):
    tsu._get_dt_original = tsu._get_dt
    tsu._get_stress_dt_original = tsu._get_stress_dt

    def _get_dt_patched(freq):
        try:
            return tsu._get_dt_original(freq)
        except Exception as exc:
            if "not Day" in str(exc) or str(freq) == "D":
                return 1.0
            raise

    def _get_stress_dt_patched(freq):
        if str(freq) == "D":
            return 1.0
        return tsu._get_stress_dt_original(freq)

    tsu._get_dt = _get_dt_patched
    tsu._get_stress_dt = _get_stress_dt_patched

    # Uppdatera modulreferenser som importerats med "from ... import _get_dt"
    ps_model._get_dt = tsu._get_dt
    ps_timeseries._get_dt = tsu._get_dt
    tsu._dayfreq_patch_applied = True

model_name = settings.get("model_name", "hindcast_model")
model_structure = settings.get("model_structure", {})
stressmodel_name = model_structure.get("stressmodel_name", "rch")

gw_model = ps.Model(ho, name=model_name)

rm1 = ps.rch.FlexModel(
    gw_uptake=model_structure.get("gw_uptake", True),
    snow=model_structure.get("snow", True)
)

sm1 = ps.RechargeModel(
    prec,
    evap_selected,
    temp=temp,
    recharge=rm1,
    rfunc=ps.Gamma(),
    name=stressmodel_name
)
gw_model.add_stressmodel(sm1)

# Om kalibreringen innehöll brusparametrar lägger vi till samma brusmodell
if params.index.to_series().astype(str).str.startswith("noise_").any():
    gw_model.add_noisemodel(ps.ArNoiseModel())

# Applicera kalibrerade parameter-värden till modellen
for pname in params.index:
    if pname in gw_model.parameters.index and "optimal" in params.columns:
        value = float(params.loc[pname, "optimal"])
        gw_model.parameters.loc[pname, "initial"] = value
        gw_model.parameters.loc[pname, "optimal"] = value
        gw_model.parameters.loc[pname, "vary"] = False

print("Modellen har byggts från sparade inställningar.")
print(gw_model.parameters[["initial", "optimal", "vary"]])

Applicerade kompatibilitetspatch för frekvens 'D'.


ValueError: Value must be Timedelta, string, integer, float, timedelta or convertible, not Day

### 3. Hindcasting

In [5]:
# Definiera hindcastperiod som överlapp i drivserier
tmin_hindcast = max(prec.index.min(), temp.index.min(), evap_selected.index.min())
tmax_hindcast = min(prec.index.max(), temp.index.max(), evap_selected.index.max())

gw_head_hindcast = gw_model.simulate(tmin=tmin_hindcast, tmax=tmax_hindcast)

# Visualisera hindcast och observationer
plt.figure(figsize=(12, 6))
plt.plot(gw_head_hindcast.index, gw_head_hindcast, label="Hindcastad grundvattennivå", color="blue")

ho_overlap = ho[tmin_hindcast:tmax_hindcast]
if len(ho_overlap) > 0:
    plt.plot(ho_overlap.index, ho_overlap, label="Observerad grundvattennivå", color="black", linewidth=1.5)

plt.title(f"Hindcasting ({evap_choice})")
plt.xlabel("Datum")
plt.ylabel("Grundvattennivå")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Hindcastperiod: {tmin_hindcast.date()} till {tmax_hindcast.date()}")
print(f"Antal hindcastade dagar: {len(gw_head_hindcast)}")

ValueError: Value must be Timedelta, string, integer, float, timedelta or convertible, not Day

### 4. Exportera hindcastingresultat

In [ ]:
safe_method = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in evap_choice)

hindcast_csv = os.path.join(data_dir, f"hindcast_{safe_method}.csv")
hindcast_xlsx = os.path.join(data_dir, f"hindcast_{safe_method}.xlsx")

gw_head_hindcast.to_csv(hindcast_csv, index=True)
gw_head_hindcast.to_excel(hindcast_xlsx, index=True)

print(f"Hindcast sparad till CSV: {hindcast_csv}")
print(f"Hindcast sparad till Excel: {hindcast_xlsx}")